# HCP LANGUAGE_LR — Spatial ICA

This notebook adapts the **spatial ICA** part of Lab 7 to the single-subject HCP `LANGUAGE_LR` run and the GLM design matrix from Part 1.

**Analysis convention:** each fMRI volume/timepoint is one sample, and each in-mask voxel is one feature:

\[
X \in \mathbb{R}^{n_{\mathrm{scans}}\times n_{\mathrm{voxels}}}.
\]

FastICA estimates spatial component maps (voxel patterns) and their associated timecourses. The notebook saves component maps, timecourses, axial-only figures, and correlations with the GLM regressors.

> Run this notebook from the same project root as the GLM notebook, with the same Python environment. The functional image and design matrix must represent the same run and the same timepoints in the same order.

## 1. Imports and project paths

The default paths follow the project structure used in Part 1. The functional image selection prefers the smoothed image used for the GLM, then the motion-corrected image, then the raw image. **If Part 1 used a different image, set `fmri_path` explicitly to that exact file.**

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
import seaborn as sns

from nilearn import image, plotting
from nilearn.maskers import NiftiMasker
from nilearn.masking import compute_epi_mask
from sklearn.decomposition import FastICA, PCA

# Project folders (same structure as Part 1)
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
FUNC_DATA_DIR = DATA_DIR / "fMRI" / "tfMRI_LANGUAGE_LR"
FUNC_PREPROC_DIR = PROJECT_ROOT / "outputs" / "functional"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "glm"
FIRST_LEVEL_DIR = OUTPUT_DIR / "first_level"
ICA_DIR = OUTPUT_DIR / "ica"
ICA_DIR.mkdir(parents=True, exist_ok=True)

DESIGN_MATRIX_PATH = OUTPUT_DIR / "design_matrix.csv"

# Prefer the final image used in the GLM. Change this explicitly if needed.
FMRI_CANDIDATES = [
    FUNC_PREPROC_DIR / "tfMRI_LANGUAGE_LR_mcf_smooth.nii.gz",
    FUNC_PREPROC_DIR / "tfMRI_LANGUAGE_LR_mcf.nii.gz",
    FUNC_DATA_DIR / "tfMRI_LANGUAGE_LR.nii.gz",
]
fmri_path = next((p for p in FMRI_CANDIDATES if p.exists()), None)

if fmri_path is None:
    raise FileNotFoundError(
        "Could not find the functional image. Set fmri_path to the same final "
        "4D image used for the GLM."
    )
if not DESIGN_MATRIX_PATH.exists():
    raise FileNotFoundError(
        f"Design matrix not found: {DESIGN_MATRIX_PATH}. Run/save tasks a-b first."
    )

print("Functional image:", fmri_path)
print("Design matrix:", DESIGN_MATRIX_PATH)
print("ICA output folder:", ICA_DIR)

## 2. Load data and verify timepoint alignment

We load the 4D image and the Part 1 design matrix. The number of rows in the design matrix must equal the number of volumes in the selected image. This check is essential: correlations in Section 6 are meaningful only if both datasets refer to exactly the same timepoints.

In [ ]:
fmri_img = nib.load(str(fmri_path))
if len(fmri_img.shape) != 4:
    raise ValueError(f"Expected a 4D fMRI image, got shape {fmri_img.shape}")

n_scans = int(fmri_img.shape[3])
zooms = fmri_img.header.get_zooms()
t_r = float(zooms[3])

design_matrix = pd.read_csv(DESIGN_MATRIX_PATH, index_col=0)
design_matrix = design_matrix.loc[
    :, ~design_matrix.columns.astype(str).str.match(r"^Unnamed:")
]
design_matrix = design_matrix.apply(pd.to_numeric, errors="raise")

if len(design_matrix) != n_scans:
    raise ValueError(
        f"Design matrix has {len(design_matrix)} rows but selected fMRI has "
        f"{n_scans} volumes. Select the exact image used in Part 1, or regenerate "
        "the design matrix for this image."
    )
if not np.isfinite(design_matrix.to_numpy(dtype=float)).all():
    raise ValueError("The design matrix contains NaN or infinite values.")

print("Image shape:", fmri_img.shape)
print(f"TR: {t_r:.4f} s")
print("Design matrix shape:", design_matrix.shape)
print("Regressors:", list(design_matrix.columns))

## 3. Define the analysis mask and extract voxel time series

We reuse the EPI mask saved during the GLM analysis when available, ensuring that ICA and GLM are evaluated in the same functional-image space. Otherwise, we estimate an EPI mask from this run.

`NiftiMasker` extracts one row per timepoint and one column per in-mask voxel. No additional smoothing or temporal detrending is applied here: the selected functional image and the GLM design matrix are used as prepared in Part 1.

In [ ]:
mask_path = FIRST_LEVEL_DIR / "epi_mask.nii.gz"
if mask_path.exists():
    mask_img = nib.load(str(mask_path))
    print("Reusing GLM EPI mask:", mask_path)
else:
    warnings.warn("GLM EPI mask not found; computing an EPI mask from this image.")
    mask_img = compute_epi_mask(fmri_img)
    mask_path = ICA_DIR / "epi_mask_ica.nii.gz"
    mask_img.to_filename(str(mask_path))

# Confirm mask and functional image occupy the same grid.
if mask_img.shape != fmri_img.shape[:3] or not np.allclose(mask_img.affine, fmri_img.affine):
    raise ValueError("EPI mask and functional image are not on the same voxel grid.")

masker = NiftiMasker(
    mask_img=mask_img,
    standardize=False,
    detrend=False,
    smoothing_fwhm=None,
)
X = masker.fit_transform(fmri_img)
X = np.asarray(X, dtype=np.float32)

if X.shape[0] != n_scans:
    raise RuntimeError("Unexpected number of extracted timepoints.")
if X.shape[1] == 0:
    raise RuntimeError("The analysis mask contains no voxels.")

print("ICA input matrix X:", X.shape, "(timepoints × voxels)")
print("Number of in-mask voxels:", X.shape[1])

## 4. Choose the number of independent components

We retain **20 components** as a practical exploratory choice for this single task run. It provides more than the 10 components required for the correlation analysis, while keeping the decomposition and visualization manageable. It is not a universal optimum: ICA dimensionality is a modeling choice, and a different component count can split or merge spatial patterns.

To make the choice more transparent, the next cell plots the cumulative variance explained by PCA. This is a diagnostic, not a claim that 20 is a statistically unique solution. If the curve suggests that 20 components capture very little of the dominant variance, or if the decomposition is unstable, consider testing nearby values (for example 15 or 25) and comparing map interpretability.

In [ ]:
N_COMPONENTS_REQUESTED = 20
n_components = min(N_COMPONENTS_REQUESTED, n_scans - 1, X.shape[1])
if n_components < 10:
    raise ValueError(
        f"Only {n_components} components are feasible, but the assignment asks "
        "for correlations of the first 10 components."
    )
if n_components != N_COMPONENTS_REQUESTED:
    warnings.warn(f"Requested 20 components; using feasible value {n_components}.")

# PCA variance diagnostic, limited to the chosen dimensionality.
pca_diagnostic = PCA(n_components=n_components, svd_solver="randomized",
                     random_state=0)
pca_diagnostic.fit(X)
cum_var = np.cumsum(pca_diagnostic.explained_variance_ratio_)

plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, n_components + 1), cum_var, marker="o")
plt.axvline(n_components, linestyle="--", color="grey",
            label=f"ICA components = {n_components}")
plt.xlabel("Number of PCA dimensions")
plt.ylabel("Cumulative explained variance")
plt.title("PCA variance diagnostic for ICA dimensionality")
plt.ylim(0, 1.02)
plt.grid(alpha=0.25)
plt.legend()
plt.show()

print(f"Cumulative variance represented by {n_components} PCA dimensions: "
      f"{cum_var[-1]:.3f}")

## 5. Run spatial FastICA

With `X` shaped as timepoints × voxels, scikit-learn treats each timepoint as a sample and each voxel as a feature. The resulting `components_` contains spatial maps (components × voxels), while `fit_transform(X)` returns their associated timecourses (timepoints × components).

FastICA component order and sign are arbitrary. Therefore, “component 1” means the first component returned by this particular fit; it is not inherently the most important or most task-related component. The fixed random seed makes the run reproducible under the same software/hardware conditions.

In [ ]:
ica = FastICA(
    n_components=n_components,
    algorithm="parallel",
    whiten="unit-variance",
    fun="logcosh",
    max_iter=1000,
    tol=1e-4,
    random_state=0,
)

# Each row of X is one timepoint/sample.
ica_timecourses = ica.fit_transform(X)       # shape: timepoints × components
ica_spatial_maps = ica.components_           # shape: components × voxels

print("ICA timecourses:", ica_timecourses.shape)
print("ICA spatial maps:", ica_spatial_maps.shape)
print("FastICA iterations:", ica.n_iter_)

if ica_timecourses.shape[0] != len(design_matrix):
    raise RuntimeError("ICA timecourses and GLM design matrix are not aligned.")

## 6. Save component maps and timecourses

Each component's voxel weights are transformed back into a 3D NIfTI image using the same masker and EPI mask. We save all estimated maps and all timecourses, not just the first 10, so the results can be inspected or reused later.

In [ ]:
component_map_paths = []
for component_idx in range(n_components):
    # inverse_transform maps the voxel vector back into the original 3D image grid.
    comp_img = masker.inverse_transform(ica_spatial_maps[component_idx, :])
    comp_path = ICA_DIR / f"ica_component_{component_idx + 1:02d}.nii.gz"
    comp_img.to_filename(str(comp_path))
    component_map_paths.append(comp_path)

timecourse_columns = [f"IC_{i + 1:02d}" for i in range(n_components)]
timecourses_df = pd.DataFrame(ica_timecourses, columns=timecourse_columns)
timecourses_path = ICA_DIR / "ica_timecourses.csv"
timecourses_df.to_csv(timecourses_path, index=False)

print(f"Saved {len(component_map_paths)} component maps in {ICA_DIR}")
print("Saved timecourses:", timecourses_path)

## 7. Display each selected component separately (axial view only)

The assignment asks for separate axial displays. The following cell plots every retained component in its own figure, with a symmetric display range based on the component's absolute voxel weights. These are ICA spatial weights, **not** GLM t- or Z-statistics; do not interpret them using GLM p-value thresholds.

In [ ]:
for component_idx, comp_path in enumerate(component_map_paths):
    comp_img = nib.load(str(comp_path))
    values = np.asarray(comp_img.dataobj)
    vmax = float(np.nanpercentile(np.abs(values), 99))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = None

    disp = plotting.plot_stat_map(
        comp_img,
        display_mode="z",       # axial slices only
        cut_coords=7,
        title=f"Spatial ICA component {component_idx + 1}",
        cmap="cold_hot",
        symmetric_cbar=True,
        vmax=vmax,
        colorbar=True,
    )
    # Save the same axial-only figure for the report.
    fig_path = ICA_DIR / f"ica_component_{component_idx + 1:02d}_axial.png"
    disp.savefig(str(fig_path), dpi=150)
    plotting.show()
    disp.close()

print("Axial component figures saved in:", ICA_DIR)

## 8. Correlate the first 10 ICA timecourses with the GLM regressors

We compute Pearson correlations between the first 10 timecourses in the order returned by FastICA and **all columns** in the saved Part 1 design matrix. This includes task regressors and any motion, spike, derivative, or drift regressors that were included when the design matrix was created.

The correlation is descriptive: it indicates linear association, not causation. The GLM design columns may be correlated with one another, and ICA component signs are arbitrary, so the sign of a component correlation should be interpreted together with its spatial map.

In [ ]:
n_ica_for_corr = min(10, ica_timecourses.shape[1])
first10_df = timecourses_df.iloc[:, :n_ica_for_corr].copy()

# Standard Pearson correlation; rows are aligned timepoints.
corr_input = pd.concat(
    [first10_df.reset_index(drop=True),
     design_matrix.reset_index(drop=True)],
    axis=1,
)
corr_all = corr_input.corr(method="pearson")
corr_matrix = corr_all.loc[first10_df.columns, design_matrix.columns]

corr_path = ICA_DIR / "ica_first10_vs_glm_regressors_correlations.csv"
corr_matrix.to_csv(corr_path)

print("Correlation matrix (first 10 ICs × all GLM regressors):")
display(corr_matrix.round(3))
print("Saved:", corr_path)

In [ ]:
plt.figure(figsize=(max(10, 0.65 * len(design_matrix.columns)), 6))
sns.heatmap(
    corr_matrix,
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    annot=(len(design_matrix.columns) <= 18),
    fmt=".2f",
    linewidths=0.25,
)
plt.xlabel("GLM design-matrix regressors")
plt.ylabel("ICA timecourses (FastICA return order)")
plt.title("Pearson correlations: first 10 ICA timecourses vs GLM regressors")
plt.xticks(rotation=60, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
heatmap_path = ICA_DIR / "ica_first10_vs_glm_regressors_heatmap.png"
plt.savefig(heatmap_path, dpi=180, bbox_inches="tight")
plt.show()
print("Saved heatmap:", heatmap_path)

## 9. Identify the strongest associations for interpretation

This table lists, for each of the first 10 components, the GLM regressor with the largest absolute correlation. Use it as a guide—not as an automatic component label. Inspect the corresponding axial spatial map before deciding whether a component resembles task-related activity, motion, a transient artifact, or a broad low-frequency pattern.

In [ ]:
strongest_rows = []
for ic_name in corr_matrix.index:
    abs_corr = corr_matrix.loc[ic_name].abs()
    best_regressor = abs_corr.idxmax()
    strongest_rows.append({
        "ICA_component": ic_name,
        "Most_correlated_GLM_regressor": best_regressor,
        "Pearson_r": corr_matrix.loc[ic_name, best_regressor],
        "Absolute_r": abs_corr[best_regressor],
    })

strongest_correlations = pd.DataFrame(strongest_rows)
strongest_path = ICA_DIR / "ica_first10_strongest_glm_correlations.csv"
strongest_correlations.to_csv(strongest_path, index=False)
display(strongest_correlations.round(3))
print("Saved:", strongest_path)

## 10. Interpretation guide and reporting

When commenting on the results, connect the correlation matrix to the corresponding component maps:

- **Task association:** a component whose timecourse correlates with `story`, `math`, or `cue` may contain signal associated with that task regressor. Check whether its spatial map is anatomically plausible and whether the association is specific or shared across task regressors.
- **Motion association:** correlations with motion parameters or their derivatives, especially alongside edge-like or peripheral spatial patterns, may indicate motion-related signal.
- **Spike association:** a strong correlation with a spike regressor can indicate that a component captures one or more outlier volumes.
- **Drift association:** correlation with cosine/high-pass drift regressors may indicate slow temporal fluctuations rather than task-specific activity.
- **Weak correlations:** do not imply that a component is meaningless. ICA finds statistically independent patterns, not components optimized to match the GLM regressors.

**Limitations to report:** this is a single-run, single-subject decomposition; ICA solutions can vary with component count, initialization, masking, and preprocessing. Component signs and ordering are arbitrary. The correlation analysis is exploratory and does not replace the GLM contrast tests or multiple-comparison correction.

### Output files

All ICA outputs are written to `outputs/glm/ica/`:

- `ica_component_XX.nii.gz` — 3D spatial map for each component.
- `ica_component_XX_axial.png` — axial-only visualization for each component.
- `ica_timecourses.csv` — all component timecourses.
- `ica_first10_vs_glm_regressors_correlations.csv` — requested correlation matrix.
- `ica_first10_vs_glm_regressors_heatmap.png` — correlation heatmap.
- `ica_first10_strongest_glm_correlations.csv` — strongest absolute correlation per component.

The GLM outputs in `outputs/glm/first_level/` are not overwritten.